# WTI Crude Oil — Exploratory Data Analysis

This notebook is a first look at the raw data behind the 5-day WTI return prediction pipeline in `../src/`. It answers three questions before any modeling happens:

1. What do the raw price series actually look like, and are there any obvious data quality issues (gaps, regime breaks, the 2020 negative-price event)?
2. What is the shape of the WTI daily return distribution — is it close to normal, or fat-tailed and skewed (as commodity returns typically are)?
3. How correlated are the candidate features with each other, and with the dollar index specifically, and does that correlation drift over time?

It reuses `src/data_loader.py` and `src/features.py` directly so the EDA is guaranteed to be looking at exactly the same data the model pipeline sees.

In [ ]:
import sys
from pathlib import Path

# make `src` importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_loader import download_prices
from src.features import compute_price_features

# Reuse the same light chart surface / ink palette as results/*.png for visual consistency.
COLOR_SURFACE = "#fcfcfb"
COLOR_GRID = "#e1e0d9"
COLOR_TEXT_PRIMARY = "#0b0b0b"
COLOR_TEXT_MUTED = "#898781"
SERIES_COLORS = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7", "#e34948", "#e87ba4"]

plt.rcParams["figure.facecolor"] = COLOR_SURFACE
plt.rcParams["axes.facecolor"] = COLOR_SURFACE
plt.rcParams["axes.edgecolor"] = COLOR_TEXT_MUTED
plt.rcParams["grid.color"] = COLOR_GRID
plt.rcParams["text.color"] = COLOR_TEXT_PRIMARY
plt.rcParams["axes.labelcolor"] = COLOR_TEXT_PRIMARY
plt.rcParams["xtick.color"] = COLOR_TEXT_MUTED
plt.rcParams["ytick.color"] = COLOR_TEXT_MUTED

In [ ]:
# Uses the cached data/prices.parquet if main.py has already been run;
# otherwise this triggers a fresh yfinance download.
prices = download_prices()
prices.tail()

## 1. Price levels

Small multiples rather than one overlaid chart, since the series live on very different scales (WTI in $/bbl vs. VIX as an index level vs. the 10Y yield in percentage points) — overlaying them on a shared or dual axis would be misleading.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 14))
axes = axes.flatten()

for ax, (col, color) in zip(axes, zip(prices.columns, SERIES_COLORS)):
    ax.plot(prices.index, prices[col], color=color, linewidth=1.1)
    ax.set_title(col, loc="left", fontweight="bold", fontsize=11)
    ax.grid(True, linewidth=0.6)
    ax.set_axisbelow(True)

axes[-1].axis("off")
fig.suptitle("Raw price/level series, 2010–present", fontsize=14, fontweight="bold", x=0.02, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

Note the April 2020 WTI collapse (COVID demand shock plus the front-month futures contract briefly trading negative) — the single largest regime break in the sample. It's a real, tradeable event rather than a data error, so it is intentionally NOT removed or winsorized anywhere in the pipeline; the walk-forward validation just has to live with it in whichever training window it falls into, exactly as a live trading system would have.

## 2. WTI daily return distribution

Financial returns are famously not normally distributed — fatter tails (more extreme moves than a Gaussian predicts) and often mild skew. This matters for modeling choice: it's part of why we favor simple, regularized models and a coarse long/short/flat strategy over anything that implicitly assumes Gaussian errors.

In [ ]:
wti_daily_logret = np.log(prices["WTI"]).diff().dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.hist(wti_daily_logret, bins=120, color=SERIES_COLORS[0], alpha=0.85)
ax1.axvline(0, color=COLOR_TEXT_MUTED, linewidth=1)
ax1.set_title("WTI daily log return distribution", loc="left", fontweight="bold")
ax1.set_xlabel("Daily log return")
ax1.grid(True, linewidth=0.6)
ax1.set_axisbelow(True)

from scipy import stats  # noqa: E402  (local import to keep the top-level dependency list minimal)
stats.probplot(wti_daily_logret, dist="norm", plot=ax2)
ax2.get_lines()[0].set_color(SERIES_COLORS[0])
ax2.get_lines()[0].set_markersize(3)
ax2.get_lines()[1].set_color(COLOR_TEXT_MUTED)
ax2.set_title("Q-Q plot vs. normal", loc="left", fontweight="bold")
ax2.grid(True, linewidth=0.6)
ax2.set_axisbelow(True)

fig.tight_layout()
plt.show()

print(f"mean:     {wti_daily_logret.mean():.5f}")
print(f"std:      {wti_daily_logret.std():.5f}")
print(f"skew:     {wti_daily_logret.skew():.3f}")
print(f"kurtosis: {wti_daily_logret.kurtosis():.3f}  (0 = normal; WTI is typically fat-tailed/leptokurtic)")

## 3. Feature correlation heatmap

Built from `compute_price_features`, i.e. the exact feature set the models train on (minus the target). High pairwise correlation between two features is a flag for redundancy — useful context when reading the Lasso/XGBoost feature importance charts in `results/`, since a model may lean on one of two correlated features somewhat arbitrarily.

In [ ]:
feats = compute_price_features(prices).dropna()
corr = feats.corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=6, color="white" if abs(corr.iloc[i, j]) > 0.6 else COLOR_TEXT_PRIMARY)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
ax.set_title("Feature correlation matrix", loc="left", fontweight="bold", pad=12)
fig.tight_layout()
plt.show()

## 4. Rolling correlation: WTI vs. the dollar index

The "stronger dollar = weaker commodities" relationship is a textbook macro link, but it isn't stable — it strengthens and weakens with the macro regime (e.g. it tends to tighten during broad risk-off/dollar-funding-stress episodes). A single full-sample correlation number would hide that, so this plots a rolling window instead.

In [ ]:
ROLLING_WINDOW = 126  # ~6 months of trading days

wti_ret = np.log(prices["WTI"]).diff()
dxy_ret = np.log(prices["DXY"]).diff()
rolling_corr = wti_ret.rolling(ROLLING_WINDOW).corr(dxy_ret)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(rolling_corr.index, rolling_corr.values, color=SERIES_COLORS[0], linewidth=1.2)
ax.axhline(0, color=COLOR_TEXT_MUTED, linewidth=1)
ax.set_title(f"Rolling {ROLLING_WINDOW}d correlation: WTI vs. DXY daily returns", loc="left", fontweight="bold")
ax.set_ylabel("Correlation")
ax.grid(True, linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print(f"Full-sample correlation: {wti_ret.corr(dxy_ret):.3f}")
print(f"Rolling correlation range: [{rolling_corr.min():.3f}, {rolling_corr.max():.3f}]")

## Takeaways for modeling

- WTI daily returns are fat-tailed relative to a normal distribution — regularized linear models and a shallow, subsampled XGBoost (see `src/models.py`) are a deliberate response to that low signal-to-noise environment, not an oversight.
- The dollar/crude relationship is regime-dependent, which is exactly why `dxy_chg_5d` / `dxy_chg_21d` are included as *changes* rather than relying on a single static correlation assumption baked into the feature.
- Several features are correlated (e.g. the momentum windows with each other, and the Brent-WTI spread level with its own 5-day change) — worth keeping in mind when interpreting `results/feature_importance.png`, since correlated features can split credit in ways that understate any one feature's true importance.